# COGS 188 - Project Proposal

# Project Description

Patrolling in the Dark: Reinforcement Learning for Safety Patrols Under Partial Observability

Our project explores how reinforcement learning can be used to optimize police patrol allocation in an environment where crime is only partially observable. We will develop a visual simulated 'city' which will be represented by a grid. Within this grid, different areas of the city will have different rates of crime depending on the time of day. The main challenge we are focusing on is observation bias. In our project, the AI agent can only observe crimes that are recorded, and crimes are much more likely to be recorded near patrol units. As a result, areas with more patrol activity may appear to have higher crime rates while areas with little to no patrol activity may appear safer than they actually are. Furthermore, the agent will continually return to areas where it has previously recorded crime, potentially reinforcing inaccurate assumptions about the city's overall crime distribution. 

By using this environment we will attempt to determine whether different reinforcement learnign algorithms and exploration strategies can overcome the bias and learn more effective patrol policies. We will compare three learning methods from the class syllabus we believed would best approach our problem: Q-learning, SARSA, and Monte Carlo. We will compare these against simple baselines such as random patrols, fixed routes, and (as mentioned previously) always going to the area with the most recorded crime. We will also test these exploration strategies: Epsilon-greedy, Upper Confidence Bound, and Optimistic Initialization, to see whether exploring less observed areas helps the agent escape its bias. Together these allow us to study on policy vs off policy learning, as well as the effect of discounting. 

Our simulated city is a 20x20 grid with three patrol units. At each time step, every unit has these action possibilities: moves up, down, left, right, or stays put. This gives us 125 combined actions. A crime is detected, meaning caught and recorded, with a probability that drops as the distance to the nearest patrol grows.The units share on policy and each shift covers a full day and night. This means early choices will affect what the agent learns later. We will use A* search to route units to reported incidents so the agent only has to learn where to patrol. 

Since this is a partially observable, non-deterministic Markov decision process, it is more complex that a typical grid world for two reasons we will be implementing. One, crime rates are set using real incident data (see the Data section). The full state is too large for a table, so it iwll be simplified to the unit's position, time of day, and summary of recently recorded crime. We will test several versions of this state, along with these hyperparameters: learning rate, discount factor, and exploration rate. Two, crime displacement will occur, meaning heavy patrolling in one area will push crime into neihboring areas. This means the environment changes as the agent acts. 

We will evaluate each method on the true crime intercepted, catch rate, the correlation bw/ recorded and true crime, how evenly patrols are spread, efficiency, and computing time. If time allows, we hope to add a fairness penalty, which punishes the agent for leaving some areas with far less patrol coverage than others. Because we know the true crime distribution in our simulation, we can check whether each method actually adapts to and understands the city. 



# Names

- Kali Anchlia
- Colin Isidro
- Melissa Wang

# Abstract 

We want to build a city represented as a grid and place a patrol somewhere that can observe and catch crime. This patrol is placed and moves around in a way that's determined by a reinforcement learning agent that tries to maximize how much crime it can catch. The data we use helps build our simulated city where we use real open incident data to decide when, where, and how frequent crime appears in our simulation in a realistic manner. Each node in our city map would have a probability of crime appearing, influenced by that data. Some of this data would be known by the agent, like the time of day and crime reported nearby. An agent also only has a probability of catching or observing a crime, where they have a general range where the further they are from the crime, the less likely they are to catch the crime and the closer they are, the more likely it is to be caught. The success of our agent will be measured by the amount of crime it is able to intercept.

# Background

In law enforcement, the timeliness of the response to an emergency plays a critical role in maintaining public safety and preventing crime, yet police response times in major U.S. cities are getting longer, partly because of staffing shortages.<a name="repasky"></a>[<sup>[1]</sup>](#repaskynote) With fewer officers available, where patrol units are positioned while on standby matters, since a well-placed patrol can reach an incident sooner. We hope to use reinforcement learning (RL) to make patrol positioning more efficient and accurate.

Prior work has applied multi-agent RL to police operations. Repasky et al., working with Atlanta Police Department data, treat each patrol unit as a Q-learning agent that shares a deep Q-network and combine it with mixed-integer programming for dispatch. Their jointly optimized policies outperformed policies optimized for patrol alone, dispatch alone, and a heuristic baseline, including in an environment built from roughly 90,000 real calls-for-service in Southwest Atlanta, though they note occasional large outliers in response time.[<sup>[1]</sup>](#repaskynote) Wong et al. use an APPO actor-critic method to reschedule existing patrol plans when incidents occur, cutting training time by a factor of 40 and improving solution quality by around 10% over an earlier RL approach. They note that the method depends on handcrafted dispatch rules, may not generalize to different environment sizes, and does not always satisfy its soft constraints.<a name="wong"></a>[<sup>[2]</sup>](#wongnote) Xu and Mao use PPO to route police vehicles through an expressway network after congestion has been detected, motivated by the fact that fixed routes and experience-based responses adapt poorly to changing conditions. Their data come from a single network, and they note that the policy can converge slowly under extreme congestion.<a name="xu"></a>[<sup>[3]</sup>](#xunote)

All three studies treat the problem as one of response and scheduling: an incident is known to the system, and the question is how quickly and with how little disruption a unit can be sent. Wong et al., for example, assume full observability and one incident at a time.[<sup>[2]</sup>](#wongnote) Our project addresses a different part of the problem: the patrol must decide where to be before crime is visible to it. We build a grid city whose crime probabilities come from real open incident data, give the agent only partial information (time of day and nearby reported crime), and let a crime be caught only with a probability that decreases with distance. This makes patrol positioning the whole problem and lets us measure success directly by the crimes the agent intercepts. Our approach is most similar to Repasky et al. in building a simulated city from real incident data, though we use different data and try different training methods.[<sup>[1]</sup>](#repaskynote)


# Problem Statement

Our problem is deciding where to send three police patrol units in a simulated 20x20 city so they intercept as much crime as possible, when the AI agent can only see crimes that get recorded near its patrols. Crime occurs at hidden rates that depend on location+time of day, and a crime is recorded with a probability of detection that drops as the nearest patrol gets farther away. Since the agent learns only from recorded crime, its own patrol choices bias its data. 

The goal is a patrol policy that maximizes the expected # of true crimes interecepted per shift. This is quantifiable because the catch rate is a formula over the crimes in an episode, and its expected value can be computed from the detection  probabilities. It is measurable because we know every true crime in the simulation, so we can score each method on identical crim sequences. It is replicable because every new run is nder the same rules, so the 'problem' occurs hundreds of times. Reinforcement learning can solve it by learning a patrol policy through repeated trial and error. 

# Data

To map a real city into our simulation grid, we will use a public crime report from the San Diego Association of Governments. This allows us to augment a "living" city that would likely have more crime rates in certain hotspots at certain times of day. 

Link to obtain data: 
https://opendata.sandag.org/ARJIS/CIBRS-Group-A-Public-Crime-Data/pr74-d3tr/about_data

This dataset has 806,000 rows and 22 columns. Each row is a single crime incident reported. 

The critical columns we will use are date/time the crime case occurred (floating timestamp), crime against category (string), zip code (string), and block address (string). 

To use this dataset, we'll first filter out the unused columns and rows with invalid/missing entries in our critical columns. We'll convert date/time just the hour as an integer from 0 to 23. We'll convert the zip code from a string to an integer. With pd.factorize(), we can convert our string block address to an integer id. Finally, we will convert the crime against category to a boolean (1 being emergency, 0 being non-patrol crimes like fraud).

We want the location variables like zip code and block address as integers to map each incident into (x,y) coordinates in our simulation grid. In terms of how we'll code the mapping, one option is first using zipcode to map incident into a 2x2 box like sudoku then using the block address id to produce specific coordinates. Our second option is using the Python library, uszipcode, to map the zip codes to the latitude/longitude then cut both coordinates to fit onto our grid. 

# Proposed Solution

In this section, clearly describe a solution to the problem. The solution should be applicable to the project domain and appropriate for the dataset(s) or input(s) given. Provide enough detail (e.g., algorithmic description and/or theoretical properties) to convince us that your solution is applicable. Why might your solution work? Make sure to describe how the solution will be tested.  

If you know details already, describe how (e.g., library used, function calls) you plan to implement the solution in a way that is reproducible.

If it is appropriate to the problem statement, describe a benchmark model<a name="sota"></a>[<sup>[3]</sup>](#sotanote) against which your solution will be compared. 

# Evaluation Metrics

Our main evaluation metric is catch rate, defined as the number of crimes caught divided by the total number of crimes that appear in an episode:

CatchRate = $ \frac{1}{N} \sum_{i=1}^{N} Y_i $

where N is the number of crimes in the episode and $Y_i$ is whether the crime was caught or not, with 1 representing caught and 0 representing not caught. For more clarity, whether a crime is caught or not is probabilistic and it depends on $d_i$, the distance between the patrol and a crime. The probability of detection, $P(d_i)$, decreases as $d_i$ grows. We want an agent that catches as many crimes as possible, so this metric directly rewards that goal, and because it is normalized by N, it keeps episodes with more crime from inflating the result and lets us compare agents on a common scale. Since detection is probabilistic, even an ideal agent will score below 1, so we interpret results relative to the benchmark. We compute the catch rate for both the benchmark model and the RL agent on identical simulated crime sequences, averaged over K episodes. Due to detection being random, we also use expected catch rate defined as:

ExpCatchRate = $\frac{1}{N} \sum_{i=1}^{N}  P(d_i) $

This removes the stochasticity of the actual outcome and replaces that with the probability of a catch for a crime rather than whether it was actually caught. It then allows us to also compare across models with a more objective, yet theoretical, measure.

# Ethics & Privacy

We considered privacy, bias, and misuse in regards to our project. The data we plan to use is a public SANDAG crime dataset, and we will drop the victim columns and block addresses so NO individual is identifiable. We will use only the area, time, and offense type to build crime rates. Reported crime reflectss where police patrol and what people report, so we will treat it only as a starting point for our simulations hidden crime rates, not as the true crime in SD. Our project is a simplified simulation and should ont be used to direct real patrols, because a tool like this could keep sending patrols to the same neighborhoods and increase over policing, which is a real ethical concern. Location can also act as a stand in for demographic groups, and since we will remove demographic data, we cannot test for this directly, which we will state as a limitation. To keep our metrics honest, we will report how evenly patrols are spread alongside catch rate, document our choices about which offenses count as patrol relevant, and clearly state our limitations. 

# Team Expectations 

Put things here that cement how you will interact/communicate as a team, how you will handle conflict and difficulty, how you will handle making decisions and setting goals/schedule, how much work you expect from each other, how you will handle deadlines, etc...
* *Team Expectation 1*
* *Team Expectation 2*
* *Team Expecation 3*
* ...

# Project Timeline Proposal

Replace this with something meaningful that is appropriate for your needs. It doesn't have to be something that fits this format.  It doesn't have to be set in stone... "no battle plan survives contact with the enemy". But you need a battle plan nonetheless, and you need to keep it updated so you understand what you are trying to accomplish, who's responsible for what, and what the expected due dates are for each item.

| Meeting Date  | Meeting Time| Completed Before Meeting  | Discuss at Meeting |
|---|---|---|---|
| 1/20  |  1 PM |  Brainstorm topics/questions (all)  | Determine best form of communication; Discuss and decide on final project topic; discuss hypothesis; begin background research | 
| 1/26  |  10 AM |  Do background research on topic (Pelé) | Discuss ideal dataset(s) and ethics; draft project proposal | 
| 2/1  | 10 AM  | Edit, finalize, and submit proposal; Search for datasets (Beckenbaur)  | Discuss Wrangling and possible analytical approaches; Assign group members to lead each specific part   |
| 2/14  | 6 PM  | Import & Wrangle Data ,do some EDA (Maradonna) | Review/Edit wrangling/EDA; Discuss Analysis Plan   |
| 2/23  | 12 PM  | Finalize wrangling/EDA; Begin programming for project (Cruyff) | Discuss/edit project code; Complete project |
| 3/13  | 12 PM  | Complete analysis; Draft results/conclusion/discussion (Carlos)| Discuss/edit full project |
| 3/19  | Before 11:59 PM  | NA | Turn in Final Project  |

# Footnotes
<a name="Repasky"></a>1.[^](#repaskynote): 
  Repasky, Matthew, et al. “Multi‐Agent Reinforcement Learning for Joint Police Patrol and Dispatch.” Naval Research Logistics, February 2026, https://doi.org/10.1002/nav.70059. <br>
<a name="wongnote"></a>2.[^](#wongnote): 
  Wong, Songhan, et al. “Dynamic Police Patrol Scheduling with Multi-Agent Reinforcement Learning.” Learning and Intelligent Optimization, edited by Meinolf Sellmann and Kevin Tierney, vol. 14286, Springer International Publishing AG, 2023, pp. 567–82, https://doi.org/10.1007/978-3-031-44505-7_38. <br>

<a name="xunote"></a>3.[^](#xunote): 
  Xu, Shuo, and Qilin Mao. “Graph Neural Network and Reinforcement Learning–Based Framework for Real-Time Traffic Congestion Detection and Police Dispatch Using Multi-Source Heterogeneous Data.” Informatica (Ljubljana), vol. 49, no. 14, November 2025, https://doi.org/10.31449/inf.v49i14.11640.<br>

